In [11]:
from BCP303.BCP303 import BPC303
from Sourcemeter.sourcemeter import Sourcemeter2401
import time
from datetime import datetime
from tool.tools import post_process


In [ ]:
# move the stage in Z direction
bcp303_z = BPC303(channel_id=1)
position_z = bcp303_z.move_to_origin(start_position=0)
step_size_z = 0.1  # move 0.1 mm
position_z = bcp303_z.bcp303_move_stage(step_size=step_size_z, current_position=position_z,)
bcp303_z.bcp303_stop(ifback=True)

In [12]:
# move the stage in x direction
bcp303_x = BPC303(channel_id=2)
position_x = bcp303_x.move_to_origin(start_position=0)
step_size_x = 0.1  # move 0.1 mm
position_x = bcp303_x.bcp303_move_stage(step_size=step_size_x, current_position=position_x,)
bcp303_x.bcp303_stop(ifback=False)

APT High Voltage Amplifier initialized
Stage Moving Done
Stage disconnected


In [ ]:
# record the signals
sm2401 = Sourcemeter2401(speed_nplc=0.1)


In [17]:
# sweep operation: move stage in z-direction and record the signals

def sweep_operation(stage_settings=None,chip_name="chip_test",sample_name="beam_test"):
    step_number = stage_settings["step_number"]
    time_interval = stage_settings["time_interval"]
    position_x = stage_settings["position_x"]
    step_size_z = stage_settings["step_size_z"]
    count = 1
    formatted_time = datetime.now().strftime("%Y%m%d%H%M")
    try:
        bcp303_z = BPC303(channel_id=1)
        bcp303_device = bcp303_z.get_device()
        # moving controller x
        bcp303_x = BPC303(channel_id=2, device=bcp303_device)
        # Sourcemeter
        sm2401 = Sourcemeter2401(speed_nplc=0.1)
        # set the initial position of the stage z
        position_z = bcp303_z.move_to_origin(start_position=stage_settings["position_z"]) - step_size_z
        time.sleep(1)
        bcp303_x.move_to_origin()
        time.sleep(1)
        allData = [{"position": [], "voltage": []}]
        for _ in range(step_number):
            position_z = bcp303_z.bcp303_move_stage(
                step_size=step_size_z,
                current_position=position_z,
            )
            allData[0]["position"].append(position_z)
            time.sleep(0.5)
            bcp303_x.move_to_position(start_position=0, step_size=0.5, final_position=position_x)
            time.sleep(0.5)
            step_start = time.perf_counter()
            voltage = sm2401.measure_voltage(duration=time_interval / 2, dt=0.01)[
                "voltage"
            ]
            allData[0]["voltage"].append(voltage)
            # remaining time to wait until the next step
            elapsed = time.perf_counter() - step_start
            remaining = time_interval - elapsed
            if remaining > 0:
                time.sleep(remaining)
            bcp303_x.move_to_origin()
            time.sleep(0.5)
            print(f"process completed: {100 * count / step_number:.1f}%")
            count += 1
        sm2401_settings = sm2401.getSettings()
        stage_settings["position_x"] = position_x
        settings = {"stage": stage_settings, "sourcemeter": sm2401_settings}
        post_process(
            chip_name=chip_name,
            sample_name=sample_name,
            result=allData[0],
            config=settings,
            repeat=1,
            position_z=position_z,
            ifshow=False,
            formatted_time=formatted_time,
        )
    # stop the stage and sourcemeter
    except Exception as e:
        print(f"Exception occurred: {e}")
    finally:
        time.sleep(0.5)
        sm2401.close()
        bcp303_z.move_to_origin()
        bcp303_z.channel.StopPolling()
        bcp303_x.bcp303_stop(ifback=True)
        return allData[0], settings
    

setting = {
        "position_x": 2,
        "step_number": 120,
        "position_z": 0,
        "step_size_z": 0.1,
        "time_interval": 2,  
    }
sweep_operation(
        stage_settings=setting,
        chip_name="V1_R_W_1_Left",
        # chip_name="SiN_beam",
        # sample_name="w2_sweep",  # test_1_right, w=20
        sample_name="test_AFM4_450_w20_4_sweep",  
    )

APT High Voltage Amplifier initialized
APT High Voltage Amplifier initialized
IDN: KEITHLEY INSTRUMENTS INC.,MODEL 2401,4614233,B02 Jan 20 2021 10:19:49/B01  /W/N
process completed: 0.8%
process completed: 1.7%
process completed: 2.5%
process completed: 3.3%
process completed: 4.2%
process completed: 5.0%
process completed: 5.8%
process completed: 6.7%
process completed: 7.5%
process completed: 8.3%
process completed: 9.2%
process completed: 10.0%
process completed: 10.8%
process completed: 11.7%
process completed: 12.5%
process completed: 13.3%
process completed: 14.2%
process completed: 15.0%
process completed: 15.8%
process completed: 16.7%
process completed: 17.5%
process completed: 18.3%
process completed: 19.2%
process completed: 20.0%
process completed: 20.8%
process completed: 21.7%
process completed: 22.5%
process completed: 23.3%
process completed: 24.2%
process completed: 25.0%
process completed: 25.8%
process completed: 26.7%
process completed: 27.5%
process completed: 28.3

({'position': [0.000610370189519944,
   0.100100711081271,
   0.198980681783502,
   0.299691763054292,
   0.399182103946043,
   0.498062074648274,
   0.596942045350505,
   0.695822016052736,
   0.791039765617847,
   0.889919736320078,
   0.989410077211829,
   1.08890041810358,
   1.18839075899533,
   1.28788109988708,
   1.38798181096835,
   1.48686178167058,
   1.58635212256233,
   1.68523209326456,
   1.78472243415632,
   1.88604388561663,
   1.98553422650838,
   2.08563493758965,
   2.18573564867092,
   2.28461561937315,
   2.3841059602649,
   2.48237556077761,
   2.58247627185888,
   2.68257698294015,
   2.7820673238319,
   2.88155766472366,
   2.98104800561541,
   3.07992797631764,
   3.18002868739891,
   3.27951902829066,
   3.37534714804529,
   3.47483748893704,
   3.57371745963927,
   3.67320780053102,
   3.77330851161229,
   3.8746299630726,
   3.97534104434339,
   4.07238990447706,
   4.17310098574786,
   4.27259132663961,
   4.37391277809992,
   4.47523422956023,
   4.575334